# Qlib Research Guide — Step-by-Step Notebook
Welcome! This notebook is a friendly, playroom-style tour of [Qlib](https://github.com/microsoft/qlib). Imagine building Lego models for stocks: Qlib gives us the blocks, instructions, and a workbench so we can test trading ideas safely.


## Part 1 — Overview of Qlib
- **What is Qlib?** It is an open-source toolbox for studying stocks. Think of it as a giant toy box full of bricks (data), blueprints (models), and testing mats (backtests) for building trading strategies.
- **Why quant researchers use it:** It helps them try ideas quickly, keep experiments tidy, and repeat results, just like re-building the same Lego house again.
- **Key concepts (kid-friendly):**
  - **Market data:** Daily prices are like the weather each day.
  - **Factors/features:** Little clues about a stock (price change, volume) — like hints in a treasure hunt.
  - **Labels:** The answer we want to learn (e.g., tomorrow's return).
  - **Models:** Smart robots that learn patterns from the clues.
  - **Alpha/signal/score:** A number from the robot saying how much it likes a stock (bigger = happier).
  - **Backtesting:** Pretend-play trading in the past to see if the idea would have worked.


## Part 2 — Installation & Setup
We install Qlib from GitHub, initialize it (either with local data you prepare or with Qlib's public providers), and check the version.


In [ ]:
# Install Qlib from the GitHub repository
# Note: uncomment the next line when running in a real environment with internet access.
# !pip install --upgrade git+https://github.com/microsoft/qlib.git

# Optional: install extra dependencies for examples (LightGBM, matplotlib, etc.)
# !pip install lightgbm matplotlib pandas ta-lib
        


In [ ]:
# Initialize Qlib with local data (assumes you already downloaded or prepared a qlib data folder)
from qlib import init as qlib_init
from qlib.utils import init_instance_by_config

# Example: point to your local qlib data directory
qlib_init(provider_uri='~/.qlib/qlib_data/cn_data', region='cn')

# For quick start using Qlib's online providers instead of local files
# qlib_init(provider_uri='~/.qlib/qlib_data', region='cn', expression_cache=None, dataset_cache=None)

# Check version
import qlib
qlib.__version__
        


## Part 3 — Data Processing
### How Qlib organizes data
- **Instruments:** Lists of stocks (e.g., all China A-share tickers) like teams of toys.
- **Calendar:** Trading days, like the days we are allowed to play.
- **Features:** The clues for each day/stock ("open/close price", "volume", etc.).
- **Labels:** The answer we want to predict (e.g., next-day return).
- **Data schema:** A tidy table with rows = (date, stock) and columns = features/labels.

### Dataset initialization
Qlib wraps data + processing rules inside a **Dataset** object, built with a **DataHandler**. The DataHandler is like a kitchen helper that slices, washes, and plates the data before cooking.


In [ ]:
# Define instruments and calendar
market = 'csi300'  # for China A-share top 300; use 'all' for all stocks
train_start, train_end = '2016-01-01', '2020-12-31'
valid_start, valid_end = '2021-01-01', '2021-12-31'
test_start, test_end = '2022-01-01', '2022-12-31'

# Describe features and labels
feature_expr = [
    '$open/Ref($close, 1) - 1',  # overnight gap
    'Mean($high, 5)/Mean($low, 5) - 1',
    'Std($volume, 10)'
]
label_expr = ['$close/Ref($close, 1) - 1']  # next-day return

# Build the DataHandler config
data_handler_config = {
    'start_time': train_start,
    'end_time': test_end,
    'instruments': market,
    'fields': feature_expr + label_expr,
    'learn_processors': [
        {'class': 'DropnaLabel'},
        {'class': 'RobustZScoreNorm', 'kwargs': {'fields_group': 'feature', 'clip_outlier': True}},
        {'class': 'Fillna', 'kwargs': {'fields_group': 'feature'}}
    ],
    'infer_processors': [
        {'class': 'RobustZScoreNorm', 'kwargs': {'fields_group': 'feature', 'clip_outlier': True}},
        {'class': 'Fillna', 'kwargs': {'fields_group': 'feature'}}
    ],
    'label': label_expr
}

# Build the Dataset config
dataset_config = {
    'class': 'DatasetH',
    'module_path': 'qlib.data.dataset',
    'kwargs': {
        'handler': data_handler_config,
        'segments': {
            'train': (train_start, train_end),
            'valid': (valid_start, valid_end),
            'test': (test_start, test_end)
        }
    }
}
        


### Downloading and loading data
Qlib ships helpers to download sample data (China A-share or US). The download is like fetching new Lego bricks.


In [ ]:
# Download daily China A-share data (requires internet; skip if already present)
# from qlib.tests.data import GetData
# GetData().qlib_data(target_dir='~/.qlib/qlib_data/cn_data', region='cn')

# Download daily US data
# GetData().qlib_data(target_dir='~/.qlib/qlib_data/us_data', region='us')

# Load raw data for a sample instrument
from qlib.data import D
df_raw = D.features(['SH600519'], ['$close', '$volume'], start_time='2022-01-01', end_time='2022-03-01')
df_raw.head()
        


### Normalizing and engineering features
Use processors to clean and scale features, and create new ones.


In [ ]:
# Initialize Dataset and preview processed data
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP

dataset = init_instance_by_config(dataset_config)
df_train = dataset.prepare('train', data_key=DataHandlerLP.DK_L)
df_train.head()
        


In [ ]:
# Plot an example feature and label
import matplotlib.pyplot as plt
sample_stock = 'SH600519'
feat_series = D.features([sample_stock], ['$close/Ref($close, 1) - 1'], start_time='2022-01-01', end_time='2022-06-01').droplevel(0)
label_series = D.features([sample_stock], label_expr, start_time='2022-01-01', end_time='2022-06-01').droplevel(0)
plt.figure(figsize=(10,4))
plt.plot(feat_series.index, feat_series.iloc[:,0], label='Feature: return-1')
plt.plot(label_series.index, label_series.iloc[:,0], label='Label: next-day return')
plt.legend()
plt.title('Sample feature and label for ' + sample_stock)
plt.show()
        


## Part 4 — Modeling
Qlib offers a **model zoo**: simple Linear models, tree models (LightGBM), and deep networks (GRU, GAT, ALSTM).
- Hyperparameters explained simply: like choosing Lego block sizes (hidden units), stack height (layers), and how long we train (epochs).
- Train/valid/test split: train = learn, valid = tune, test = final exam.


In [ ]:
# Define a LightGBM baseline model
lgb_model_config = {
    'class': 'LGBModel',
    'module_path': 'qlib.contrib.model.gbdt',
    'kwargs': {
        'loss': 'mse',
        'learning_rate': 0.05,
        'num_leaves': 64,
        'n_estimators': 200,
        'max_depth': 6
    }
}

# Define a GRU deep model
gru_model_config = {
    'class': 'GRUModel',
    'module_path': 'qlib.contrib.model.pytorch_gru',
    'kwargs': {
        'd_feat': len(feature_expr),
        'hidden_size': 64,  # size of the Lego bricks inside the network
        'num_layers': 2,    # how many brick layers
        'dropout': 0.1,
        'n_epochs': 30,
        'lr': 1e-3
    }
}

# Pick one model to train
model_config = lgb_model_config
model = init_instance_by_config(model_config)

# Train with train/valid split
model.fit(dataset)

# Save and load model
model.save('model.pkl')
model_loaded = init_instance_by_config(model_config)
model_loaded.load('model.pkl')
        


## Part 5 — Forecasting / Inference
We feed test data to the trained model, get scores (alpha), and visualize them.
- **Alpha/signal/score:** The model's happiness score for each stock-day (bigger = more excited).


In [ ]:
# Prepare test data and predict
test_ds = dataset.prepare('test', data_key=DataHandlerLP.DK_I)
pred = model_loaded.predict(test_ds)
pred.head()

# Save predictions
pred.to_csv('predictions.csv')

# Visualize prediction distribution
plt.figure(figsize=(6,4))
plt.hist(pred.values, bins=50, alpha=0.7)
plt.title('Prediction score distribution')
plt.xlabel('Score (alpha)')
plt.ylabel('Frequency')
plt.show()
        


## Part 6 — Portfolio & Backtest
We turn scores into trades and test them in a sandbox.
- **Risk model:** Safety rails that limit how wild trades can be.
- **TrainerFlow / Workflow:** The recipe combining data, model, and backtest steps.
- **Execution model:** How orders are filled (like waiting in a checkout line).
- **Position management:** How many shares we hold.
- **Sharpe ratio:** How much reward per unit of wobble (higher is better).
- **Drawdown:** Biggest fall from a peak, like sliding down a playground slide.
- **Turnover:** How often we swap toys (trades).


In [ ]:
# Set up backtest config
port_config = {
    'benchmark': 'SH000905',  # small-cap China CSI 500 index
    'risk_model': None,
    'order_generator': {
        'class': 'OrderGen',
        'kwargs': {
            'strategy': {
                'class': 'TopkDropoutStrategy',
                'module_path': 'qlib.contrib.strategy',
                'kwargs': {
                    'signal': pred,
                    'topk': 20,
                    'n_drop': 5
                }
            },
            'risk_model': None
        }
    },
    'execution': {
        'class': 'SimExecution',
        'module_path': 'qlib.backtest.execution'
    }
}

from qlib.workflow import R
rec = R.start(exp_name='demo_backtest')
backtest_result = R.backtest(pred, port_config)
report, positions = backtest_result
R.save_objects(trade_decision=pred, positions=positions)
R.end_recorder()

report.head()
        


In [ ]:
# Plot equity curve
equity = report['return'].cumsum().apply(lambda x: (1 + x))
plt.figure(figsize=(10,4))
plt.plot(equity.index, equity.values, label='Strategy')
plt.axhline(1.0, color='gray', linestyle='--')
plt.legend()
plt.title('Backtest equity curve')
plt.show()
        


## Part 7 — Performance Analysis
We score how good the predictions and backtest were.
- **IC (Information Coefficient):** Correlation between scores and real returns — like checking if happy faces matched candy size.
- **ICIR:** Stability of IC (average divided by its wiggle).
- **Cumulative/Annualized return:** How the money pile grew over time.
- **Risk metrics:** Sharpe (reward vs wobble), max drawdown (biggest slide), volatility.
- **Confusion matrix-like view:** Group scores into buckets and see how returns behaved.
- **Feature importance:** For tree models, which clues mattered most.


In [ ]:
# Prediction evaluation
from qlib.contrib.evaluate import risk_analysis
from qlib.contrib.evaluate import indicator_analysis

analysis_df = risk_analysis(pred, dataset.prepare('test', col_set=['label']))
analysis_df.head()

# IC/ICIR
ic = analysis_df['ic'].mean()
icir = ic / analysis_df['ic'].std()
print('IC:', ic, 'ICIR:', icir)

# Feature importance for LightGBM model
if hasattr(model, 'feature_importances_'):
    importance = model.feature_importances_
    for f, score in zip(feature_expr, importance):
        print(f, score)

# Confusion-matrix-like bucket analysis
indicator_analysis(report['return'], by=pred)
        


## Part 8 — Advanced Qlib Features
- **Online serving:** Turn your model into a small server that returns scores for new data.
- **Optimization & pipelines:** Automate training + backtesting with workflows.
- **Custom DataHandlers:** Your own kitchen helper for exotic data.
- **Custom models:** Plug in any PyTorch/LightGBM model via a simple template.


In [ ]:
# Online serving sketch
from qlib.serving import configs as serving_configs
serv_conf = serving_configs.MODEL_SERV_CONF
serv_conf['model_uri'] = 'model.pkl'
# In practice, run: qlib_serving server --config_path path/to/conf.yml

# Custom DataHandler template
from qlib.data.dataset.handler import DataHandler

class MyHandler(DataHandler):
    def setup_data(self, *args, **kwargs):
        # load or generate your own data tables here
        pass

# Custom PyTorch model template
import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self, d_feat):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_feat, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

# Wrap the model for Qlib (sketch)
# from qlib.contrib.model.pytorch_utils import PytorchModel
# class MyQlibModel(PytorchModel):
#     def __init__(self, **kwargs):
#         super().__init__(net=MyModel(kwargs['d_feat']), **kwargs)

# Workflow pipeline example (auto train + backtest)
from qlib.workflow.online.utils import OnlineManager
online_mgr = OnlineManager(model_uri='model.pkl', dataset=dataset)
# online_mgr.train_and_update()
        


## Part 9 — Summary & Next Steps
- Explore more factors (technical, fundamental).
- Try US datasets (`region='us'`).
- Experiment with advanced models (GAT, Transformer-based).
- Read Qlib docs: https://qlib.readthedocs.io
- Contribute: open issues/PRs on GitHub, share new DataHandlers or models.
- Next: connect to live paper-trading or build a dashboard for your signals.
